# VoxCPM LoRA Fine-tuning (Arabic walkthrough)

This notebook mirrors the `examples/finetune_voxcpm_lora.py` script and walks through the
major steps required to adapt VoxCPM to an Arabic single-speaker dataset using LoRA.
Run the cells sequentially after filling in your Hugging Face credentials and dataset
paths.

> **Tip:** The defaults assume the public [`arbml/arabic_single_speaker_speech_dataset`](https://huggingface.co/datasets/arbml/arabic_single_speaker_speech_dataset).
> Verify access from your environment and switch to another dataset if needed.
> Always resample to 16 kHz mono to match VoxCPM's AudioVAE expectations.

In [ ]:
# Optional: install libraries when running in a fresh notebook environment (e.g. Google Colab).
# Uncomment if required.
# !pip install --upgrade pip
# !pip install datasets huggingface_hub accelerate torchaudio

In [ ]:
import os
from dataclasses import dataclass

HF_TOKEN = os.getenv("HF_TOKEN", "hf_your_token_here")
MODEL_ID = "openbmb/VoxCPM-0.5B"
DATASET_ID = "arbml/arabic_single_speaker_speech_dataset"
DATASET_CONFIG = None  # e.g. a subset name if the dataset exposes multiple configs
DATASET_SPLIT = "train"
TEXT_COLUMN = "text"
AUDIO_COLUMN = "audio"
CACHE_DIR = os.getenv("VOXCPM_CACHE", None)
OUTPUT_DIR = "./voxcpm-arabic-lora"
MAX_SAMPLES = 64  # reduce for dry-runs / debugging
BATCH_SIZE = 2
LEARNING_RATE = 1e-4
DIFFUSION_STEPS = 10
STOP_LOSS_WEIGHT = 0.1
TRAINING_STEPS = 100  # adjust based on compute budget
LORA_RANK = 16
LORA_ALPHA = 32.0
LORA_DROPOUT = 0.05
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0

In [ ]:
from huggingface_hub import login

if HF_TOKEN and HF_TOKEN != "hf_your_token_here":
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    print("Skipping hub login; set HF_TOKEN for private assets.")

In [ ]:
from datasets import Audio, load_dataset

dataset = load_dataset(
    path=DATASET_ID,
    name=DATASET_CONFIG,
    split=DATASET_SPLIT,
    cache_dir=CACHE_DIR,
    use_auth_token=HF_TOKEN if HF_TOKEN != "hf_your_token_here" else None,
)

dataset = dataset.cast_column(AUDIO_COLUMN, Audio(sampling_rate=16000))
if MAX_SAMPLES:
    dataset = dataset.select(range(min(len(dataset), MAX_SAMPLES)))

dataset[0]

In [ ]:
import torch
from tqdm.auto import tqdm

from voxcpm import VoxCPM
from voxcpm.model.utils import get_dtype

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pipeline = VoxCPM.from_pretrained(
    hf_model_id=MODEL_ID,
    load_denoiser=False,
    cache_dir=CACHE_DIR,
)
model = pipeline.tts_model.to(device)
model.audio_vae.to(device)
model.audio_vae.eval()
model.audio_vae.requires_grad_(False)
model.requires_grad_(False)

lm_dtype = get_dtype(model.config.dtype)
model.base_lm.setup_cache(BATCH_SIZE, model.config.max_length, model.device, lm_dtype)
model.residual_lm.setup_cache(BATCH_SIZE, model.config.max_length, model.device, lm_dtype)

In [ ]:
from examples.finetune_voxcpm_lora import (
    VoxCPMArabicDataset,
    build_dataloader,
    replace_with_lora,
    teacher_forced_loss,
    save_lora_adapters,
)

target_suffixes = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "enc_to_lm_proj",
    "lm_to_dit_proj",
    "res_to_dit_proj",
    "stop_proj",
]
matched = replace_with_lora(
    model,
    target_suffixes=target_suffixes,
    rank=LORA_RANK,
    alpha=LORA_ALPHA,
    dropout=LORA_DROPOUT,
)
print(f"Injected LoRA adapters into {len(matched)} modules.")

In [ ]:
wrapped_dataset = VoxCPMArabicDataset(
    dataset=dataset,
    model=model,
    text_column=TEXT_COLUMN,
    audio_column=AUDIO_COLUMN,
    target_sr=16000,
)
dataloader = build_dataloader(wrapped_dataset, model=model, batch_size=BATCH_SIZE)
len(dataloader)

In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

global_step = 0
model.train()
progress = tqdm(total=TRAINING_STEPS, desc="LoRA fine-tuning")

for batch in dataloader:
    optimizer.zero_grad()
    loss = teacher_forced_loss(
        model,
        batch,
        diffusion_steps=DIFFUSION_STEPS,
        stop_loss_weight=STOP_LOSS_WEIGHT,
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
    optimizer.step()

    global_step += 1
    progress.update(1)
    progress.set_postfix({"loss": loss.item()})

    if global_step >= TRAINING_STEPS:
        break

progress.close()
print(f"Finished {global_step} optimization steps.")

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
save_lora_adapters(model, OUTPUT_DIR)
print(f"Saved LoRA weights to {OUTPUT_DIR}")

## Next steps
- Merge the adapters with the base VoxCPM checkpoint during inference or load them dynamically with PEFT-style utilities.
- Evaluate on a held-out validation set by reusing the batching logic above and monitoring speech quality.
- Iterate on `TRAINING_STEPS`, `LORA_RANK`, and other hyper-parameters as you expand your dataset.